# 01 - Bronze EDA

## Input Paths

- `s3a://bronze/yellow_taxi/`
- `s3a://bronze/green_taxi/`
- `s3a://bronze/weather/`

## Setup Spark Và Utilities

In [1]:
from pathlib import Path
import json
import sys

from pyspark.sql.functions import avg, col, count as spark_count, length, lit, max as spark_max, min as spark_min, percentile_approx, sum as spark_sum, when

cwd = Path.cwd().resolve()
if (cwd / "utils").exists():
    notebooks_dir = cwd
elif (cwd / "notebooks" / "utils").exists():
    notebooks_dir = cwd / "notebooks"
else:
    notebooks_dir = cwd

if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

from utils.spark_session import (
    BRONZE_GREEN_PATH,
    BRONZE_WEATHER_PATH,
    BRONZE_YELLOW_PATH,
    get_spark,
    path_exists,
    safe_display,
    show_schema,
)

spark = get_spark("MetroPulse 01 Bronze EDA")
print(f"Spark version: {spark.version}")
print(f"Spark timezone: {spark.conf.get('spark.sql.session.timeZone')}")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /tmp/metropulse-notebook-ivy-20260526/cache
The jars for the packages stored in: /tmp/metropulse-notebook-ivy-20260526/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2276b982-ac46-487d-b7d9-b2eb0292a128;1.0
	confs: [default]


	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 304ms :: artifacts dl 11ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-2276b982-ac46-487d-b7d9-b2eb0292a128
	confs: [default]

26/05/26 03:16:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark version: 3.5.1


Spark timezone: America/New_York


## Phạm Vi Dữ Liệu Bronze

Ba path Bronze được đối chiếu trực tiếp trên MinIO. Dataset đọc được trong snapshot này xác định phạm vi của các số liệu volume, metadata và raw payload được trình bày bên dưới.

In [2]:
bronze_sources = [
    {
        "name": "yellow_taxi",
        "path": BRONZE_YELLOW_PATH,
        "expected_fields": ["tpep_pickup_datetime", "VendorID", "PULocationID"],
    },
    {
        "name": "green_taxi",
        "path": BRONZE_GREEN_PATH,
        "expected_fields": ["lpep_pickup_datetime", "trip_type"],
    },
    {
        "name": "weather",
        "path": BRONZE_WEATHER_PATH,
        "expected_fields": ["timestamp", "temperature_f", "precipitation_mm"],
    },
]

available_datasets = {}

for source in bronze_sources:
    name = source["name"]
    path = source["path"]
    try:
        if not path_exists(spark, path):
            print(f"Cảnh báo: Không tìm thấy Bronze path cho {name}: {path}. Bỏ qua dataset này.")
            continue
        available_datasets[name] = spark.read.parquet(path)
        print(f"Đã đọc Bronze dataset: {name} -> {path}")
    except Exception as exc:
        print(f"Cảnh báo: Không thể đọc {name} tại {path}. Lý do: {exc}")

if not available_datasets:
    print("Cảnh báo: Chưa có Bronze dataset nào khả dụng. Hãy chạy producer và `make bronze` trước khi phân tích sâu.")

26/05/26 03:16:37 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Đã đọc Bronze dataset: yellow_taxi -> s3a://bronze/yellow_taxi/


Đã đọc Bronze dataset: green_taxi -> s3a://bronze/green_taxi/


Đã đọc Bronze dataset: weather -> s3a://bronze/weather/


## Chỉ Tiêu Kiểm Chứng Bronze

Các phép đo tập trung vào số event, metadata Kafka, duplicate offset và kích thước payload. Đây là những chỉ tiêu trực tiếp chứng minh khả năng audit của Bronze mà không biến đổi bản ghi nguồn.

In [3]:
metadata_columns = [
    "topic",
    "partition",
    "offset",
    "kafka_timestamp",
    "key",
    "json_data",
    "ingestion_timestamp",
]

dimension_candidates = ["ingestion_date", "topic", "taxi_type", "data_type"]


def missing_columns(df, columns):
    existing = set(df.columns)
    return [column_name for column_name in columns if column_name not in existing]


def show_missing_columns(df, columns, label):
    missing = missing_columns(df, columns)
    if missing:
        print(f"{label} missing columns: {', '.join(missing)}")
    else:
        print(f"{label}: all requested columns are present.")
    return missing


def show_counts_by_dimensions(df, dataset_name):
    for dimension in dimension_candidates:
        if dimension not in df.columns:
            print(f"{dataset_name}: thiếu column `{dimension}`, bỏ qua count theo dimension này.")
            continue
        print(f"{dataset_name}: row count by `{dimension}`")
        counts_df = df.groupBy(dimension).agg(spark_count(lit(1)).alias("row_count")).orderBy(col(dimension))
        safe_display(counts_df, n=100, truncate=False)


def show_metadata_null_profile(df, dataset_name):
    existing_columns = [column_name for column_name in metadata_columns if column_name in df.columns]
    missing = show_missing_columns(df, metadata_columns, f"{dataset_name} Kafka metadata")
    if not existing_columns:
        print(f"{dataset_name}: không có metadata column nào để kiểm tra null.")
        return

    total_rows = df.count()
    null_row = df.agg(
        *[
            spark_sum(when(col(column_name).isNull(), 1).otherwise(0)).alias(column_name)
            for column_name in existing_columns
        ]
    ).collect()[0]

    profile_rows = []
    for column_name in existing_columns:
        null_count = int(null_row[column_name] or 0)
        null_ratio = (null_count / total_rows) if total_rows else 0.0
        profile_rows.append((column_name, null_count, null_ratio, "present"))

    for column_name in missing:
        profile_rows.append((column_name, None, None, "missing"))

    profile_df = spark.createDataFrame(profile_rows, ["column_name", "null_count", "null_ratio", "status"])
    safe_display(profile_df, n=len(profile_rows), truncate=False)


def show_duplicate_offset_check(df, dataset_name):
    required = ["topic", "partition", "offset"]
    missing = missing_columns(df, required)
    if missing:
        print(f"{dataset_name}: thiếu {missing}, không thể kiểm tra duplicate Kafka offsets.")
        return

    duplicate_groups = (
        df.groupBy("topic", "partition", "offset")
        .agg(spark_count(lit(1)).alias("row_count"))
        .where(col("row_count") > 1)
    )
    duplicate_group_count = duplicate_groups.count()
    print(f"{dataset_name}: duplicate Kafka offset group count = {duplicate_group_count}")
    if duplicate_group_count > 0:
        safe_display(duplicate_groups.orderBy(col("row_count").desc()), n=20, truncate=False)


def get_one_json_sample(df, dataset_name):
    if "json_data" not in df.columns:
        print(f"{dataset_name}: thiếu `json_data`, không thể xem raw payload.")
        return None

    rows = df.select("json_data").where(col("json_data").isNotNull()).limit(1).collect()
    if not rows:
        print(f"{dataset_name}: không có dòng `json_data` non-null để inspection.")
        return None

    sample = rows[0]["json_data"]
    print(f"{dataset_name}: one raw json_data sample")
    print(sample[:2000])
    return sample


def inspect_json_payload(sample, expected_fields, dataset_name):
    if sample is None:
        return
    try:
        parsed = json.loads(sample)
    except Exception as exc:
        print(f"{dataset_name}: không parse được JSON sample. Lý do: {exc}")
        return

    if isinstance(parsed, dict):
        keys = sorted(parsed.keys())
        print(f"{dataset_name}: parsed JSON keys sample ({len(keys)} keys):")
        print(keys[:80])
        comparison_rows = [
            (field_name, field_name in parsed, parsed.get(field_name) if field_name in parsed else None)
            for field_name in expected_fields
        ]
        comparison_df = spark.createDataFrame(comparison_rows, ["expected_field", "is_present", "sample_value"])
        safe_display(comparison_df, n=len(comparison_rows), truncate=False)
    else:
        print(f"{dataset_name}: JSON sample không phải object/dict, type={type(parsed).__name__}")

## Schema, Sample Và Raw Payload

Schema, preview giới hạn và một payload raw cho mỗi nguồn cho thấy Bronze đang lưu event envelope cùng `json_data` nguyên bản. Việc đọc mẫu chỉ nhằm quan sát cấu trúc, không tạo schema đã chuẩn hóa tại Bronze.

In [4]:
for source in bronze_sources:
    dataset_name = source["name"]
    df = available_datasets.get(dataset_name)
    if df is None:
        print(f"\n=== {dataset_name} ===")
        print(f"Cảnh báo: Dataset {dataset_name} không khả dụng, bỏ qua schema/sample/raw payload.")
        continue

    print(f"\n=== {dataset_name}: schema ===")
    show_schema(df)

    print(f"\n=== {dataset_name}: small sample ===")
    safe_display(df.limit(10), n=10, truncate=False)

    print(f"\n=== {dataset_name}: raw json_data sample ===")
    sample = get_one_json_sample(df, dataset_name)
    inspect_json_payload(sample, source["expected_fields"], dataset_name)


=== yellow_taxi: schema ===
root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- kafka_timestamp_type: integer (nullable = true)
 |-- key: string (nullable = true)
 |-- json_data: string (nullable = true)
 |-- taxi_type: string (nullable = false)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- ingestion_date: date (nullable = true)


=== yellow_taxi: small sample ===


+---------------+---------+--------+-----------------------+--------------------+----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+-----------------------+--------------+
|topic          |partition|offset  |kafka_timestamp        |kafka_timestamp_type|key |json_data                                                                                                                                                                               

yellow_taxi: one raw json_data sample
{"VendorID": 2, "tpep_pickup_datetime": "2024-04-03T06:51:01", "tpep_dropoff_datetime": "2024-04-03T07:01:33", "passenger_count": null, "trip_distance": 1.99, "RatecodeID": null, "store_and_fwd_flag": null, "PULocationID": 4, "DOLocationID": 68, "payment_type": 0, "fare_amount": 14.77, "extra": 0.0, "mta_tax": 0.5, "tip_amount": 2.0, "tolls_amount": 0.0, "improvement_surcharge": 1.0, "total_amount": 20.77, "congestion_surcharge": null, "Airport_fee": null, "_taxi_type": "yellow", "_source_file": "yellow_tripdata_2024-04.parquet", "_ingestion_timestamp": "2026-05-12T10:48:01.923554"}
yellow_taxi: parsed JSON keys sample (22 keys):
['Airport_fee', 'DOLocationID', 'PULocationID', 'RatecodeID', 'VendorID', '_ingestion_timestamp', '_source_file', '_taxi_type', 'congestion_surcharge', 'extra', 'fare_amount', 'improvement_surcharge', 'mta_tax', 'passenger_count', 'payment_type', 'store_and_fwd_flag', 'tip_amount', 'tolls_amount', 'total_amount', 'tpep_dro

+--------------------+----------+-------------------+
|expected_field      |is_present|sample_value       |
+--------------------+----------+-------------------+
|tpep_pickup_datetime|true      |2024-04-03T06:51:01|
|VendorID            |true      |2                  |
|PULocationID        |true      |4                  |
+--------------------+----------+-------------------+


=== green_taxi: schema ===
root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- kafka_timestamp_type: integer (nullable = true)
 |-- key: string (nullable = true)
 |-- json_data: string (nullable = true)
 |-- taxi_type: string (nullable = false)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- ingestion_date: date (nullable = true)


=== green_taxi: small sample ===



[Stage 8:===================>                                       (1 + 2) / 3]



...

green_taxi: one raw json_data sample
{"VendorID": 2, "lpep_pickup_datetime": "2023-01-01T00:26:10", "lpep_dropoff_datetime": "2023-01-01T00:37:11", "store_and_fwd_flag": "N", "RatecodeID": 1.0, "PULocationID": 166, "DOLocationID": 143, "passenger_count": 1.0, "trip_distance": 2.58, "fare_amount": 14.9, "extra": 1.0, "mta_tax": 0.5, "tip_amount": 4.03, "tolls_amount": 0.0, "ehail_fee": null, "improvement_surcharge": 1.0, "total_amount": 24.18, "payment_type": 1.0, "trip_type": 1.0, "congestion_surcharge": 2.75, "_taxi_type": "green", "_source_file": "green_tripdata_2023-01.parquet", "_ingestion_timestamp": "2026-05-12T08:29:53.052057"}
green_taxi: parsed JSON keys sample (23 keys):
['DOLocationID', 'PULocationID', 'RatecodeID', 'VendorID', '_ingestion_timestamp', '_source_file', '_taxi_type', 'congestion_surcharge', 'ehail_fee', 'extra', 'fare_amount', 'improvement_surcharge', 'lpep_dropoff_datetime', 'lpep_pickup_datetime', 'mta_tax', 'passenger_count', 'payment_type', 'store_and_fwd_f

+--------------------+----------+-------------------+
|expected_field      |is_present|sample_value       |
+--------------------+----------+-------------------+
|lpep_pickup_datetime|true      |2023-01-01T00:26:10|
|trip_type           |true      |1.0                |
+--------------------+----------+-------------------+


=== weather: schema ===
root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- kafka_timestamp_type: integer (nullable = true)
 |-- key: string (nullable = true)
 |-- json_data: string (nullable = true)
 |-- data_type: string (nullable = false)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- ingestion_date: date (nullable = true)


=== weather: small sample ===


+--------------+---------+------+-----------------------+--------------------+----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+-----------------------+--------------+
|topic         |partition|offset|kafka_timestamp        |kafka_timestamp_type|key |json_data                                                                                                                                                                                                                                                                                                                                                                                           

weather: one raw json_data sample
{"timestamp": "2023-01-01T00:00", "latitude": 40.7128, "longitude": -74.006, "location": "NYC_Manhattan", "temperature_f": 51.5, "humidity_percent": 99, "precipitation_mm": 1.0, "weather_code": 55, "weather_description": "Dense drizzle", "wind_speed_kmh": 11.3, "wind_direction_deg": 239, "cloud_cover_percent": 100, "_ingestion_timestamp": "2026-05-12 08:29:40.111110", "_source": "open_meteo"}
weather: parsed JSON keys sample (14 keys):
['_ingestion_timestamp', '_source', 'cloud_cover_percent', 'humidity_percent', 'latitude', 'location', 'longitude', 'precipitation_mm', 'temperature_f', 'timestamp', 'weather_code', 'weather_description', 'wind_direction_deg', 'wind_speed_kmh']


+----------------+----------+----------------+
|expected_field  |is_present|sample_value    |
+----------------+----------+----------------+
|timestamp       |true      |2023-01-01T00:00|
|temperature_f   |true      |51.5            |
|precipitation_mm|true      |1.0             |
+----------------+----------+----------------+



## Row Counts Theo Dimension Có Sẵn

Tôi tổng hợp số dòng theo các dimension thực sự có trong dữ liệu như `ingestion_date`, `topic`, `taxi_type` hoặc `data_type`. Cách tổng hợp này cho thấy phân bố ingestion và hỗ trợ đọc incremental theo partition.

In [5]:
for dataset_name, df in available_datasets.items():
    print(f"\n=== {dataset_name}: row counts ===")
    show_counts_by_dimensions(df, dataset_name)


=== yellow_taxi: row counts ===
yellow_taxi: row count by `ingestion_date`



[Stage 17:==>                                                     (4 + 4) / 107]




[Stage 17:=======>                                               (14 + 4) / 107]




[Stage 17:=============>                                         (27 + 4) / 107]




[Stage 17:========================>                              (48 + 4) / 107]



...

+--------------+---------+
|topic         |row_count|
+--------------+---------+
|nyc_taxi_green|1447278  |
+--------------+---------+

green_taxi: row count by `taxi_type`


+---------+---------+
|taxi_type|row_count|
+---------+---------+
|green    |1447278  |
+---------+---------+

green_taxi: thiếu column `data_type`, bỏ qua count theo dimension này.

=== weather: row counts ===
weather: row count by `ingestion_date`


+--------------+---------+
|ingestion_date|row_count|
+--------------+---------+
|2026-05-12    |17544    |
+--------------+---------+

weather: row count by `topic`


+--------------+---------+
|topic         |row_count|
+--------------+---------+
|weather_stream|17544    |
+--------------+---------+

weather: thiếu column `taxi_type`, bỏ qua count theo dimension này.
weather: row count by `data_type`


+---------+---------+
|data_type|row_count|
+---------+---------+
|weather  |17544    |
+---------+---------+



## Kafka Metadata Null Profile

Null profile được lập cho các trường truy vết gồm `topic`, `partition`, `offset`, `kafka_timestamp`, `key`, `json_data` và `ingestion_timestamp`.

In [6]:
for dataset_name, df in available_datasets.items():
    print(f"\n=== {dataset_name}: Kafka metadata null profile ===")
    show_metadata_null_profile(df, dataset_name)


=== yellow_taxi: Kafka metadata null profile ===
yellow_taxi Kafka metadata: all requested columns are present.



[Stage 44:======================>                                (43 + 4) / 107]




[Stage 44:==========================================>            (82 + 4) / 107]




[Stage 47:=======>                                               (15 + 4) / 107]




[Stage 47:===========>                                           (23 + 4) / 107]



...


[Stage 47:=================================================>     (96 + 4) / 107]




[Stage 47:====================================================> (104 + 3) / 107]



+-------------------+----------+----------+-------+
|column_name        |null_count|null_ratio|status |
+-------------------+----------+----------+-------+
|topic              |0         |0.0       |present|
|partition          |0         |0.0       |present|
|offset             |0         |0.0       |present|
|kafka_timestamp    |0         |0.0       |present|
|key                |107580599 |1.0       |present|
|json_data          |0         |0.0       |present|
|ingestion_timestamp|0         |0.0       |present|
+-------------------+----------+----------+-------+


=== green_taxi: Kafka metadata null profile ===
green_taxi Kafka metadata: all requested columns are present.


+-------------------+----------+----------+-------+
|column_name        |null_count|null_ratio|status |
+-------------------+----------+----------+-------+
|topic              |0         |0.0       |present|
|partition          |0         |0.0       |present|
|offset             |0         |0.0       |present|
|kafka_timestamp    |0         |0.0       |present|
|key                |1447278   |1.0       |present|
|json_data          |0         |0.0       |present|
|ingestion_timestamp|0         |0.0       |present|
+-------------------+----------+----------+-------+


=== weather: Kafka metadata null profile ===
weather Kafka metadata: all requested columns are present.


+-------------------+----------+----------+-------+
|column_name        |null_count|null_ratio|status |
+-------------------+----------+----------+-------+
|topic              |0         |0.0       |present|
|partition          |0         |0.0       |present|
|offset             |0         |0.0       |present|
|kafka_timestamp    |0         |0.0       |present|
|key                |17544     |1.0       |present|
|json_data          |0         |0.0       |present|
|ingestion_timestamp|0         |0.0       |present|
+-------------------+----------+----------+-------+



## Duplicate Kafka Offset Check

Tính truy vết của Kafka dựa trên bộ khóa `topic`, `partition`, `offset`. Kết quả kiểm tra snapshot hiện tại cho thấy không phát hiện nhóm offset trùng ở cả ba nguồn Bronze.

In [7]:
for dataset_name, df in available_datasets.items():
    print(f"\n=== {dataset_name}: duplicate Kafka offset check ===")
    show_duplicate_offset_check(df, dataset_name)


=== yellow_taxi: duplicate Kafka offset check ===



[Stage 68:>                                                       (1 + 4) / 107]




[Stage 68:==>                                                     (5 + 4) / 107]




[Stage 68:===>                                                    (7 + 4) / 107]




[Stage 68:====>                                                   (9 + 4) / 107]



...


[Stage 70:============================>                            (5 + 4) / 10]




[Stage 70:=======================================>                 (7 + 3) / 10]



yellow_taxi: duplicate Kafka offset group count = 0

=== green_taxi: duplicate Kafka offset check ===


green_taxi: duplicate Kafka offset group count = 0

=== weather: duplicate Kafka offset check ===


weather: duplicate Kafka offset group count = 0


## Evidence Summary: Volume, Metadata Và Payload Size

Snapshot hiện tại gồm `107,580,599` yellow taxi events, `1,447,278` green taxi events và `17,544` weather events; không có duplicate Kafka offset groups. Payload trung bình lần lượt là `582.26`, `594.75` và `389.41` ký tự, phù hợp với cách truyền JSON event ở quy mô prototype.

In [8]:
evidence_rows = []
for dataset_name, df in available_datasets.items():
    required = [column_name for column_name in metadata_columns if column_name != "key" and column_name in df.columns]
    aggregations = [spark_count(lit(1)).alias("event_rows")]
    aggregations.extend(
        spark_sum(when(col(column_name).isNull(), 1).otherwise(0)).alias(f"null_{column_name}")
        for column_name in required
    )
    if "json_data" in df.columns:
        aggregations.extend([
            spark_min(length("json_data")).alias("min_payload_chars"),
            avg(length("json_data")).alias("avg_payload_chars"),
            percentile_approx(length("json_data"), 0.95).alias("p95_payload_chars"),
            spark_max(length("json_data")).alias("max_payload_chars"),
        ])
    result = df.agg(*aggregations).first().asDict()
    metadata_nulls = sum(int(result.get(f"null_{column_name}") or 0) for column_name in required)
    duplicate_groups = None
    if all(column_name in df.columns for column_name in ["topic", "partition", "offset"]):
        duplicate_groups = (
            df.groupBy("topic", "partition", "offset")
            .count()
            .where(col("count") > 1)
            .count()
        )
    evidence_rows.append((
        dataset_name,
        int(result["event_rows"]),
        metadata_nulls,
        duplicate_groups,
        result.get("min_payload_chars"),
        float(result["avg_payload_chars"]) if result.get("avg_payload_chars") is not None else None,
        result.get("p95_payload_chars"),
        result.get("max_payload_chars"),
    ))

if evidence_rows:
    bronze_evidence_df = spark.createDataFrame(
        evidence_rows,
        ["dataset_name", "event_rows", "required_metadata_nulls", "duplicate_offset_groups", "min_payload_chars", "avg_payload_chars", "p95_payload_chars", "max_payload_chars"],
    )
    safe_display(bronze_evidence_df, n=len(evidence_rows), truncate=False)
else:
    print("Không có Bronze dataset để sinh evidence summary.")


[Stage 86:===>                                                    (7 + 4) / 107]




[Stage 86:=====>                                                 (10 + 4) / 107]




[Stage 86:=======>                                               (15 + 4) / 107]




[Stage 86:========>                                              (17 + 4) / 107]




[Stage 86:===========>                                           (22 + 4) / 107]



...


[Stage 89:===================================================>  (102 + 4) / 107]




[Stage 89:====================================================> (105 + 2) / 107]




[Stage 91:===========>                                             (2 + 4) / 10]




[Stage 91:============================>                            (5 + 4) / 10]



+------------+----------+-----------------------+-----------------------+-----------------+------------------+-----------------+-----------------+
|dataset_name|event_rows|required_metadata_nulls|duplicate_offset_groups|min_payload_chars|avg_payload_chars |p95_payload_chars|max_payload_chars|
+------------+----------+-----------------------+-----------------------+-----------------+------------------+-----------------+-----------------+
|yellow_taxi |107580599 |0                      |0                      |574              |582.2567385500429 |591              |601              |
|green_taxi  |1447278   |0                      |0                      |587              |594.7490661780253 |607              |618              |
|weather     |17544     |0                      |0                      |385              |389.40840173278616|395              |398              |
+------------+----------+-----------------------+-----------------------+-----------------+------------------+--------

## Payload Structure Comparison

- Yellow taxi: `tpep_pickup_datetime`, `VendorID`, `PULocationID`
- Green taxi: `lpep_pickup_datetime`, `trip_type`
- Weather: `timestamp`, `temperature_f`, `precipitation_mm`

Sự khác biệt này giải thích vì sao bước parse và chuẩn hóa schema phải diễn ra ở Silver; Bronze vẫn bảo toàn JSON gốc để phục vụ audit.

In [9]:
payload_summary_rows = []

for source in bronze_sources:
    dataset_name = source["name"]
    df = available_datasets.get(dataset_name)
    if df is None:
        for field_name in source["expected_fields"]:
            payload_summary_rows.append((dataset_name, field_name, None, "dataset_missing"))
        continue

    sample = get_one_json_sample(df, dataset_name)
    if sample is None:
        for field_name in source["expected_fields"]:
            payload_summary_rows.append((dataset_name, field_name, None, "sample_missing"))
        continue

    try:
        parsed = json.loads(sample)
        for field_name in source["expected_fields"]:
            payload_summary_rows.append((dataset_name, field_name, field_name in parsed, "checked"))
    except Exception as exc:
        for field_name in source["expected_fields"]:
            payload_summary_rows.append((dataset_name, field_name, None, f"parse_failed: {str(exc)[:120]}"))

payload_summary_df = spark.createDataFrame(
    payload_summary_rows,
    ["dataset_name", "expected_field", "is_present_in_sample", "status"],
)
safe_display(payload_summary_df, n=len(payload_summary_rows), truncate=False)

yellow_taxi: one raw json_data sample
{"VendorID": 2, "tpep_pickup_datetime": "2024-04-03T06:51:01", "tpep_dropoff_datetime": "2024-04-03T07:01:33", "passenger_count": null, "trip_distance": 1.99, "RatecodeID": null, "store_and_fwd_flag": null, "PULocationID": 4, "DOLocationID": 68, "payment_type": 0, "fare_amount": 14.77, "extra": 0.0, "mta_tax": 0.5, "tip_amount": 2.0, "tolls_amount": 0.0, "improvement_surcharge": 1.0, "total_amount": 20.77, "congestion_surcharge": null, "Airport_fee": null, "_taxi_type": "yellow", "_source_file": "yellow_tripdata_2024-04.parquet", "_ingestion_timestamp": "2026-05-12T10:48:01.923554"}


green_taxi: one raw json_data sample
{"VendorID": 2, "lpep_pickup_datetime": "2023-01-01T00:26:10", "lpep_dropoff_datetime": "2023-01-01T00:37:11", "store_and_fwd_flag": "N", "RatecodeID": 1.0, "PULocationID": 166, "DOLocationID": 143, "passenger_count": 1.0, "trip_distance": 2.58, "fare_amount": 14.9, "extra": 1.0, "mta_tax": 0.5, "tip_amount": 4.03, "tolls_amount": 0.0, "ehail_fee": null, "improvement_surcharge": 1.0, "total_amount": 24.18, "payment_type": 1.0, "trip_type": 1.0, "congestion_surcharge": 2.75, "_taxi_type": "green", "_source_file": "green_tripdata_2023-01.parquet", "_ingestion_timestamp": "2026-05-12T08:29:53.052057"}
weather: one raw json_data sample
{"timestamp": "2023-01-01T00:00", "latitude": 40.7128, "longitude": -74.006, "location": "NYC_Manhattan", "temperature_f": 51.5, "humidity_percent": 99, "precipitation_mm": 1.0, "weather_code": 55, "weather_description": "Dense drizzle", "wind_speed_kmh": 11.3, "wind_direction_deg": 239, "cloud_cover_percent": 100, "_inge

+------------+--------------------+--------------------+-------+
|dataset_name|expected_field      |is_present_in_sample|status |
+------------+--------------------+--------------------+-------+
|yellow_taxi |tpep_pickup_datetime|true                |checked|
|yellow_taxi |VendorID            |true                |checked|
|yellow_taxi |PULocationID        |true                |checked|
|green_taxi  |lpep_pickup_datetime|true                |checked|
|green_taxi  |trip_type           |true                |checked|
|weather     |timestamp           |true                |checked|
|weather     |temperature_f       |true                |checked|
|weather     |precipitation_mm    |true                |checked|
+------------+--------------------+--------------------+-------+



## Business Interpretation

- Snapshot Bronze đã lưu thành công ba nguồn raw với quy mô rất khác nhau: yellow taxi chiếm phần lớn volume, trong khi weather có volume nhỏ phù hợp với dữ liệu theo giờ.
- Việc không phát hiện duplicate Kafka offset groups củng cố khả năng truy vết ingestion của snapshot đang phân tích; Bronze vẫn giữ vai trò lưu nguyên event thay vì áp dụng quy tắc loại bỏ nghiệp vụ.
- Kích thước payload trung bình dưới khoảng `600` ký tự cho ba nguồn cho thấy event JSON còn gọn trong phạm vi prototype, dù kết luận throughput production cần kiểm thử tải riêng.
- Do taxi và weather có field timestamp và schema raw khác nhau, tôi chuyển việc chuẩn hóa kiểu dữ liệu và join theo local hour `America/New_York` sang Silver.
- Partition `ingestion_date` hỗ trợ các lần đọc theo batch/ngày mà không buộc Spark scan lại toàn bộ raw history.